<a href="https://colab.research.google.com/github/himanshusar123/-ClaimVision-AI/blob/main/Day_22_EY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install groq
from google.colab import userdata
import os

groq_key = userdata.get("GROQ_API_KEY")

if not groq_key:
    raise ValueError(
        "GROQ_API_KEY was not found. Add it in Colab Secrets and enable notebook access."
    )

os.environ["GROQ_API_KEY"] = groq_key
print("Groq API key loaded securely from Colab Secrets.")


Groq API key loaded securely from Colab Secrets.


In [2]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Model availability can change. Confirm the model ID in your Groq console.
MODEL_NAME = "openai/gpt-oss-120b"

print("Groq client is ready.")

Groq client is ready.


In [3]:
from google.colab import userdata
import os
from groq import Groq

MODEL_API_KEY = userdata.get("GROQ_API_KEY")
if not MODEL_API_KEY:
    raise ValueError("Add GROQ_API_KEY in Google Colab Secrets.")

os.environ["GROQ_API_KEY"] = MODEL_API_KEY
MODEL_API_URL = "Groq SDK chat-completions interface"

client = Groq(api_key=MODEL_API_KEY)

print("Groq API configured successfully.")
print("Provider endpoint will be called through the Groq Python SDK.")

Groq API configured successfully.
Provider endpoint will be called through the Groq Python SDK.


In [4]:
feedback_data = [
    {
        "feedback_id": "FB1001",
        "channel": "Online",
        "rating": 2,
        "comment": "My order arrived three days late and the delivery person did not call."
    },
    {
        "feedback_id": "FB1002",
        "channel": "Store",
        "rating": 5,
        "comment": "The staff was helpful and checkout was very quick."
    },
    {
        "feedback_id": "FB1003",
        "channel": "Online",
        "rating": 1,
        "comment": "I received a damaged product and nobody has responded to my complaint."
    },
    {
        "feedback_id": "FB1004",
        "channel": "Mobile App",
        "rating": 3,
        "comment": "The app is useful but payment failed twice before the order went through."
    },
    {
        "feedback_id": "FB1005",
        "channel": "Store",
        "rating": 4,
        "comment": "Good product quality. I would like more offers for returning customers."
    }
]

print("Total feedback records:", len(feedback_data))

for item in feedback_data:
    print(item)

Total feedback records: 5
{'feedback_id': 'FB1001', 'channel': 'Online', 'rating': 2, 'comment': 'My order arrived three days late and the delivery person did not call.'}
{'feedback_id': 'FB1002', 'channel': 'Store', 'rating': 5, 'comment': 'The staff was helpful and checkout was very quick.'}
{'feedback_id': 'FB1003', 'channel': 'Online', 'rating': 1, 'comment': 'I received a damaged product and nobody has responded to my complaint.'}
{'feedback_id': 'FB1004', 'channel': 'Mobile App', 'rating': 3, 'comment': 'The app is useful but payment failed twice before the order went through.'}
{'feedback_id': 'FB1005', 'channel': 'Store', 'rating': 4, 'comment': 'Good product quality. I would like more offers for returning customers.'}


In [5]:
import pandas as pd

feedback_df = pd.DataFrame(feedback_data)
display(feedback_df)

,feedback_id,channel,rating,comment
0,FB1001,Online,2,My order arrived three days late and the deliv...
1,FB1002,Store,5,The staff was helpful and checkout was very qu...
2,FB1003,Online,1,I received a damaged product and nobody has re...
3,FB1004,Mobile App,3,The app is useful but payment failed twice bef...
4,FB1005,Store,4,Good product quality. I would like more offers...


In [6]:
import json

SYSTEM_PROMPT = """
You are a retail customer-experience analyst.
Analyse the customer feedback provided by the user.
Return only a valid JSON object with these keys:
- sentiment: one of Positive, Neutral, Negative
- category: one of Delivery, Product Quality, Staff, Payment, App Experience, Pricing, Other
- summary: one concise sentence describing the issue or praise
- recommended_action: one practical action for the retail team
Do not invent facts. If information is unclear, say so briefly.
Do not include Markdown fences or text outside the JSON object.
"""

def analyse_feedback(feedback):
    user_content = f"""
Feedback ID: {feedback['feedback_id']}
Channel: {feedback['channel']}
Rating: {feedback['rating']}/5
Customer comment: {feedback['comment']}
Analyse this feedback.
"""

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ]
    )

    raw_text = completion.choices[0].message.content.strip()
    result = json.loads(raw_text)

    required_keys = {
        "sentiment", "category", "summary", "recommended_action"
    }
    if not required_keys.issubset(result):
        raise ValueError("Model response is missing one or more required JSON fields.")

    return result

test_result = analyse_feedback(feedback_data[0])
print(json.dumps(test_result, indent=2, ensure_ascii=False))

{
  "sentiment": "Negative",
  "category": "Delivery",
  "summary": "The order arrived three days late and the delivery person failed to call the customer.",
  "recommended_action": "Investigate the delay, improve delivery scheduling, and ensure drivers contact customers before arrival."
}


In [7]:
analysed_records = []

for feedback in feedback_data:
    try:
        analysis = analyse_feedback(feedback)
        record = {**feedback, **analysis, "processing_status": "processed"}
    except Exception as exc:
        record = {
            **feedback,
            "sentiment": None,
            "category": None,
            "summary": None,
            "recommended_action": None,
            "processing_status": "failed",
            "error": str(exc)
        }

    analysed_records.append(record)
    print(feedback["feedback_id"], "->", record["processing_status"])

analysed_df = pd.DataFrame(analysed_records)
display(analysed_df)

FB1001 -> processed
FB1002 -> processed
FB1003 -> processed
FB1004 -> processed
FB1005 -> processed


,feedback_id,channel,rating,comment,sentiment,category,summary,recommended_action,processing_status
0,FB1001,Online,2,My order arrived three days late and the deliv...,Negative,Delivery,The order arrived three days late and the deli...,"Investigate the delay, improve delivery schedu...",processed
1,FB1002,Store,5,The staff was helpful and checkout was very qu...,Positive,Staff,The staff were helpful and checkout was very q...,Maintain current staff training and ensure che...,processed
2,FB1003,Online,1,I received a damaged product and nobody has re...,Negative,Product Quality,Customer received a damaged product and has no...,"Immediately contact the customer, arrange a re...",processed
3,FB1004,Mobile App,3,The app is useful but payment failed twice bef...,Neutral,Payment,"The mobile app is useful, but the payment proc...",Investigate and resolve the payment processing...,processed
4,FB1005,Store,4,Good product quality. I would like more offers...,Positive,Pricing,Customer appreciates product quality but wants...,Introduce a loyalty program or special discoun...,processed


In [8]:
from pathlib import Path

OUTPUT_FILE = Path("/content/retail_feedback_analysis.jsonl")

with OUTPUT_FILE.open("w", encoding="utf-8") as file:
    for record in analysed_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\\n")

print("Saved:", OUTPUT_FILE)
print("Records:", len(analysed_records))

Saved: /content/retail_feedback_analysis.jsonl
Records: 5


In [10]:
with OUTPUT_FILE.open("w", encoding="utf-8") as file:
    for record in analysed_records:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

print("JSONL file recreated successfully.")

JSONL file recreated successfully.


In [11]:
loaded_records = []

with OUTPUT_FILE.open("r", encoding="utf-8") as file:
    for line in file:
        if line.strip():
            loaded_records.append(json.loads(line))

print("Records read:", len(loaded_records))

display(pd.DataFrame(loaded_records))

Records read: 5


,feedback_id,channel,rating,comment,sentiment,category,summary,recommended_action,processing_status
0,FB1001,Online,2,My order arrived three days late and the deliv...,Negative,Delivery,The order arrived three days late and the deli...,"Investigate the delay, improve delivery schedu...",processed
1,FB1002,Store,5,The staff was helpful and checkout was very qu...,Positive,Staff,The staff were helpful and checkout was very q...,Maintain current staff training and ensure che...,processed
2,FB1003,Online,1,I received a damaged product and nobody has re...,Negative,Product Quality,Customer received a damaged product and has no...,"Immediately contact the customer, arrange a re...",processed
3,FB1004,Mobile App,3,The app is useful but payment failed twice bef...,Neutral,Payment,"The mobile app is useful, but the payment proc...",Investigate and resolve the payment processing...,processed
4,FB1005,Store,4,Good product quality. I would like more offers...,Positive,Pricing,Customer appreciates product quality but wants...,Introduce a loyalty program or special discoun...,processed


In [12]:
from pathlib import Path

PROJECT_DIR = Path("/content/retail-feedback-ai")

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder created:", PROJECT_DIR)

Project folder created: /content/retail-feedback-ai


In [14]:

%%writefile /content/retail-feedback-ai/main.py

import os
import json
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from groq import Groq


# --------------------------------------------------
# 1. CREATE FASTAPI APPLICATION
# --------------------------------------------------

app = FastAPI(
    title="Retail Feedback Intelligence API",
    description="Analyse retail customer feedback using Groq.",
    version="1.0.0"
)


# --------------------------------------------------
# 2. GROQ CONFIGURATION
# --------------------------------------------------

MODEL_NAME = os.getenv(
    "GROQ_MODEL",
    "openai/gpt-oss-120b"
)

API_KEY = os.getenv("GROQ_API_KEY")

client = Groq(api_key=API_KEY) if API_KEY else None


# --------------------------------------------------
# 3. JSONL DATA STORAGE
# --------------------------------------------------

DATA_FILE = Path(
    os.getenv(
        "DATA_FILE",
        "/app/retail_feedback_analysis.jsonl"
    )
)

DATA_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# 4. INPUT VALIDATION
# --------------------------------------------------

class FeedbackRequest(BaseModel):

    feedback_id: str = Field(
        min_length=2,
        max_length=40
    )

    channel: str = Field(
        min_length=2,
        max_length=40
    )

    rating: int = Field(
        ge=1,
        le=5
    )

    comment: str = Field(
        min_length=5,
        max_length=2000
    )


# --------------------------------------------------
# 5. LLM SYSTEM PROMPT
# --------------------------------------------------

SYSTEM_PROMPT = (
    "You are a retail customer-experience analyst. "
    "Return only valid JSON with keys sentiment, category, summary, "
    "recommended_action. Sentiment must be Positive, Neutral or Negative. "
    "Category must be Delivery, Product Quality, Staff, Payment, "
    "App Experience, Pricing or Other. Do not invent facts."
)


# --------------------------------------------------
# 6. ANALYSE CUSTOMER FEEDBACK USING GROQ
# --------------------------------------------------

def run_analysis(payload):

    if client is None:
        raise RuntimeError(
            "GROQ_API_KEY is not configured."
        )

    user_text = (
        f"Feedback ID: {payload.feedback_id}\n"
        f"Channel: {payload.channel}\n"
        f"Rating: {payload.rating}/5\n"
        f"Comment: {payload.comment}"
    )

    response = client.chat.completions.create(

        model=MODEL_NAME,

        temperature=0.2,

        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_text
            }
        ]
    )

    raw_text = response.choices[0].message.content.strip()

    parsed = json.loads(raw_text)

    required = {
        "sentiment",
        "category",
        "summary",
        "recommended_action"
    }

    if not required.issubset(parsed):
        raise ValueError(
            "LLM response is missing required fields."
        )

    return parsed


# --------------------------------------------------
# 7. SAVE ANALYSIS INTO JSONL FILE
# --------------------------------------------------

def save_record(record):

    with DATA_FILE.open(
        "a",
        encoding="utf-8"
    ) as file:

        file.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )


# --------------------------------------------------
# 8. HEALTH CHECK ENDPOINT
# --------------------------------------------------

@app.get("/health")
def health():

    return {
        "status": "ok",
        "service": "retail-feedback-intelligence"
    }


# --------------------------------------------------
# 9. ANALYSE FEEDBACK ENDPOINT
# --------------------------------------------------

@app.post("/analyse")
def analyse(request: FeedbackRequest):

    request_id = str(uuid4())

    try:

        result = run_analysis(request)

        record = {
            "request_id": request_id,
            **request.model_dump(),
            **result,
            "status": "processed",
            "created_at": datetime.now(
                timezone.utc
            ).isoformat()
        }

        save_record(record)

        return record

    except Exception as exc:

        raise HTTPException(
            status_code=502,
            detail=f"Analysis failed: {type(exc).__name__}"
        )


# --------------------------------------------------
# 10. RETRIEVE ALL FEEDBACK RECORDS
# --------------------------------------------------

@app.get("/feedback")
def list_feedback():

    if not DATA_FILE.exists():

        return {
            "records": [],
            "count": 0
        }

    records = []

    with DATA_FILE.open(
        "r",
        encoding="utf-8"
    ) as file:

        for line in file:

            if line.strip():

                records.append(
                    json.loads(line)
                )

    return {
        "records": records,
        "count": len(records)
    }


# --------------------------------------------------
# 11. RETRIEVE ONE FEEDBACK RECORD
# --------------------------------------------------

@app.get("/feedback/{request_id}")
def get_feedback(request_id: str):

    if DATA_FILE.exists():

        with DATA_FILE.open(
            "r",
            encoding="utf-8"
        ) as file:

            for line in file:

                if line.strip():

                    record = json.loads(line)

                    if record.get("request_id") == request_id:

                        return record

    raise HTTPException(
        status_code=404,
        detail="Record not found"
    )


Overwriting /content/retail-feedback-ai/main.py


In [15]:
%%writefile /content/retail-feedback-ai/requirements.txt

fastapi
uvicorn[standard]
pydantic
groq

Writing /content/retail-feedback-ai/requirements.txt


In [16]:
%%writefile /content/retail-feedback-ai/Dockerfile

FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]

Writing /content/retail-feedback-ai/Dockerfile


In [17]:
import shutil

shutil.copy2(
    "/content/retail_feedback_analysis.jsonl",
    "/content/retail-feedback-ai/retail_feedback_analysis.jsonl"
)

print("JSONL dataset copied successfully.")

JSONL dataset copied successfully.


In [18]:
import os

for file in os.listdir(PROJECT_DIR):
    print(file)

main.py
retail_feedback_analysis.jsonl
Dockerfile
requirements.txt


In [19]:
import shutil
from pathlib import Path

ZIP_BASE = "/content/retail-feedback-ai-project"

zip_path = shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir="/content",
    base_dir="retail-feedback-ai"
)

print("ZIP created successfully:", zip_path)
print("ZIP size:", Path(zip_path).stat().st_size, "bytes")

ZIP created successfully: /content/retail-feedback-ai-project.zip
ZIP size: 3516 bytes


In [20]:
from google.colab import files

files.download("/content/retail-feedback-ai-project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
!curl -sL https://aka.ms/InstallAzureCLIDeb | bash

Get:1 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:6 http://archive.ubuntu.com/ubuntu noble InRelease
Get:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,116 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,552 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.0 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-backports InRelease 

In [23]:
!az version

{
  "azure-cli": "2.92.0",
  "azure-cli-core": "2.92.0",
  "azure-cli-telemetry": "1.1.0",
  "extensions": {}
}


In [24]:
!az login --use-device-code

To sign in, use a web browser to open the page https://login.microsoft.com/device and enter the code OMTHK6U2U to authenticate.

Retrieving tenants and subscriptions for the selection...

[Tenant and subscription selection]

No     Subscription name     Subscription ID                       Tenant
-----  --------------------  ------------------------------------  -----------------------------------
[1] *  Azure subscription 1  337f2b3a-68b6-4a2e-befd-01a13f20c1d0  Synergific Software Private Limited

The default is marked with an *; the default tenant is 'Synergific Software Private Limited' and subscription is 'Azure subscription 1' (337f2b3a-68b6-4a2e-befd-01a13f20c1d0).

Select a subscription and tenant (Type a number or Enter for no changes): 1

Tenant: Synergific Software Private Limited
Subscription: Azure subscription 1 (337f2b3a-68b6-4a2e-befd-01a13f20c1d0)

[Announcements]
With the new Azure CLI login experience, you can select the subscription you want to use more easily. Lea

In [25]:
!az account show --output table

EnvironmentName    HomeTenantId                          IsDefault    Name                  State    TenantDefaultDomain                     TenantDisplayName                    TenantId
-----------------  ------------------------------------  -----------  --------------------  -------  --------------------------------------  -----------------------------------  ------------------------------------
AzureCloud         0e22b77c-7ae6-4b55-bf27-94f160f78abb  True         Azure subscription 1  Enabled  infosynergificsoftware.onmicrosoft.com  Synergific Software Private Limited  0e22b77c-7ae6-4b55-bf27-94f160f78abb


In [26]:
RESOURCE_GROUP = "rg-retail-feedback-ai"
LOCATION = "centralindia"

ACR_NAME = "retailfeedbackacr2026"
AKS_NAME = "aks-retail-feedback"

IMAGE_NAME = "retail-feedback-ai"
IMAGE_TAG = "1.0"

print("Azure deployment variables configured successfully")

Azure deployment variables configured successfully


In [28]:
!az group list --output table

Name                                  Location    Status
------------------------------------  ----------  ---------
tpl-azure-aks--mazeazusrqn9-99uz-sbx  southindia  Succeeded


In [29]:
RESOURCE_GROUP = "tpl-azure-aks--mazeazusrqn9-99uz-sbx"
LOCATION = "southindia"

ACR_NAME = "retailfeedbackacr2026"
AKS_NAME = "aks-retail-feedback"

IMAGE_NAME = "retail-feedback-ai"
IMAGE_TAG = "1.0"

print("Using existing Azure Resource Group:", RESOURCE_GROUP)
print("Location:", LOCATION)

Using existing Azure Resource Group: tpl-azure-aks--mazeazusrqn9-99uz-sbx
Location: southindia


In [30]:
!az group show \
    --name "$RESOURCE_GROUP" \
    --output table

Location    Name
----------  ------------------------------------
southindia  tpl-azure-aks--mazeazusrqn9-99uz-sbx


In [31]:
!az resource list \
    --resource-group "$RESOURCE_GROUP" \
    --output table

In [32]:
import json
import subprocess

result = subprocess.run(
    [
        "az", "resource", "list",
        "--resource-group", RESOURCE_GROUP,
        "--output", "json"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    resources = json.loads(result.stdout)

    print("Total existing resources:", len(resources))

    if resources:
        for resource in resources:
            print(
                resource["name"],
                "|", resource["type"],
                "|", resource["location"]
            )
    else:
        print("No existing resources found in this Resource Group.")
else:
    print(result.stderr)

Total existing resources: 0
No existing resources found in this Resource Group.


In [33]:
ACR_NAME = "retailfeedbackacr2026"

!az acr check-name \
    --name "$ACR_NAME" \
    --output table

AvailableLoginServerName          NameAvailable
--------------------------------  ---------------
retailfeedbackacr2026.azurecr.io  True


In [34]:
!az acr create \
    --resource-group "$RESOURCE_GROUP" \
    --name "$ACR_NAME" \
    --location "$LOCATION" \
    --sku Basic \
    --admin-enabled false

{
  "adminUserEnabled": false,
  "anonymousPullEnabled": false,
  "autoGeneratedDomainNameLabelScope": "Unsecure",
  "creationDate": "2026-11-04T05:12:03.837791+00:00",
  "dataEndpointEnabled": false,
  "dataEndpointHostNames": [],
  "encryption": {
    "keyVaultProperties": null,
    "status": "disabled"
  },
  "endpointProtocol": "IPv4",
  "id": "/subscriptions/337f2b3a-68b6-4a2e-befd-01a13f20c1d0/resourceGroups/tpl-azure-aks--mazeazusrqn9-99uz-sbx/providers/Microsoft.ContainerRegistry/registries/retailfeedbackacr2026",
  "identity": null,
  "location": "southindia",
  "loginServer": "retailfeedbackacr2026.azurecr.io",
  "metadataSearch": "Disabled",
  "name": "retailfeedbackacr2026",
  "networkRuleBypassAllowedForTasks": false,
  "networkRuleBypassOptions": "AzureServices",
  "networkRuleSet": null,
  "policies": {
    "azureAdAuthenticationAsArmPolicy": {
      "status": "enabled"
    },
    "exportPolicy": {
      "status": "enabled"
    },
    "quarantinePolicy": {
      "status"

In [35]:
!az acr show \
    --resource-group "$RESOURCE_GROUP" \
    --name "$ACR_NAME" \
    --output table

NAME                   RESOURCE GROUP                        LOCATION    SKU    LOGIN SERVER                      CREATION DATE         ADMIN ENABLED
---------------------  ------------------------------------  ----------  -----  --------------------------------  --------------------  ---------------
retailfeedbackacr2026  tpl-azure-aks--mazeazusrqn9-99uz-sbx  southindia  Basic  retailfeedbackacr2026.azurecr.io  2026-11-04T05:12:03Z  False


In [36]:
import os

PROJECT_DIR = "/content/retail-feedback-ai"

print("Project folder exists:", os.path.exists(PROJECT_DIR))

if os.path.exists(PROJECT_DIR):
    print(os.listdir(PROJECT_DIR))

Project folder exists: True
['main.py', 'retail_feedback_analysis.jsonl', 'Dockerfile', 'requirements.txt']


In [37]:
print(os.listdir(PROJECT_DIR))
!docker --version

['main.py', 'retail_feedback_analysis.jsonl', 'Dockerfile', 'requirements.txt']
/bin/bash: line 1: docker: command not found


In [38]:
!az acr show \
    --name "$ACR_NAME" \
    --resource-group "$RESOURCE_GROUP" \
    --query "{name:name,loginServer:loginServer,sku:sku.name}" \
    --output table

Registry names may contain only alpha numeric characters and must be between 5 and 50 characters


In [39]:
import os
import zipfile

PROJECT_DIR = "/content/retail-feedback-ai"
ZIP_PATH = "/content/retail-feedback-ai.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(PROJECT_DIR):
        filepath = os.path.join(PROJECT_DIR, filename)

        if os.path.isfile(filepath):
            zipf.write(filepath, arcname=filename)

print("Project ZIP created:", ZIP_PATH)
print("Files:", os.listdir(PROJECT_DIR))

Project ZIP created: /content/retail-feedback-ai.zip
Files: ['main.py', 'retail_feedback_analysis.jsonl', 'Dockerfile', 'requirements.txt']


In [40]:
import zipfile
import os

BUILD_DIR = "/content/retail-feedback-build"

os.makedirs(BUILD_DIR, exist_ok=True)

with zipfile.ZipFile("/content/retail-feedback-ai.zip", "r") as zipf:
    zipf.extractall(BUILD_DIR)

print(os.listdir(BUILD_DIR))

['main.py', 'retail_feedback_analysis.jsonl', 'Dockerfile', 'requirements.txt']


In [41]:
!az acr build \
    --registry "$ACR_NAME" \
    --image "$IMAGE_NAME:$IMAGE_TAG" \
    --file "$BUILD_DIR/Dockerfile" \
    "$BUILD_DIR"

Packing source code into tar to upload...
Uploading archived source code from '/tmp/build_archive_5ba03e98236545ac97b32d601419aadd.tar.gz'...
Sending context (3.051 KiB) to registry: retailfeedbackacr2026...
Queued a build with ID: ct1
Waiting for an agent...
2026/11/04 05:36:12 Downloading source code...
2026/11/04 05:36:13 Finished downloading source code
2026/11/04 05:36:13 Using acb_vol_df99d75d-76c0-452e-84aa-95290e020bba as the home volume
2026/11/04 05:36:13 Setting up Docker configuration...
2026/11/04 05:36:13 Successfully set up Docker configuration
2026/11/04 05:36:13 Logging in to registry: retailfeedbackacr2026.azurecr.io
2026/11/04 05:36:14 Successfully logged into retailfeedbackacr2026.azurecr.io
2026/11/04 05:36:14 Executing step ID: build. Timeout(sec): 28800, Working directory: '', Network: ''
2026/11/04 05:36:14 Scanning for dependencies...
2026/11/04 05:36:14 Successfully scanned dependencies
2026/11/04 05:36:14 Launching container with name: build
Sending build con

In [42]:
!az acr repository list \
    --name "$ACR_NAME" \
    --output table

Result
------------------
retail-feedback-ai


In [43]:
!az acr repository show-tags \
    --name "$ACR_NAME" \
    --repository "$IMAGE_NAME" \
    --output table

Result
--------
1.0


In [44]:
IMAGE_URI = f"{ACR_NAME}.azurecr.io/{IMAGE_NAME}:{IMAGE_TAG}"

print("Azure Container Image:", IMAGE_URI)

Azure Container Image: retailfeedbackacr2026.azurecr.io/retail-feedback-ai:1.0


In [45]:
!az aks list \
    --resource-group "$RESOURCE_GROUP" \
    --output table

In [46]:
RESOURCE_GROUP = "tpl-azure-aks--mazeazusrqn9-99uz-sbx"

In [47]:
!az aks list \
    --resource-group "$RESOURCE_GROUP" \
    --output table

In [48]:
!az aks list --output table

In [49]:
!az provider show \
    --namespace Microsoft.ContainerService \
    --query "registrationState" \
    --output tsv

Registered


In [50]:
!az vm list-usage \
    --location southindia \
    --output table

Name                                      CurrentValue    Limit
----------------------------------------  --------------  -------
Total Regional Low-priority vCPUs         98              6000
Total Regional vCPUs                      0               18350
Virtual Machines                          21              25000
Availability Sets                         0               2500
Virtual Machine Scale Sets                0               2500
Dedicated vCPUs                           0               3000
Cloud Services                            0               2500
Basic A Family vCPUs                      0               50
Standard A0-A7 Family vCPUs               0               50
Standard A8-A11 Family vCPUs              0               50
Standard Av2 Family vCPUs                 0               50
Standard BS Family vCPUs                  0               5000
Standard Basv2 Family vCPUs               0               50
Standard Bsv2 Family vCPUs                0               5

In [51]:
!az role assignment list \
    --assignee "$(az ad signed-in-user show --query id -o tsv)" \
    --scope "/subscriptions/$(az account show --query id -o tsv)/resourceGroups/$RESOURCE_GROUP" \
    --output table

Principal                                       Role                               Scope
----------------------------------------------  ---------------------------------  -------------------------------------------------------------------------------------------------------
sb-mazeazusrqn9t23-ui4w@synergificsoftware.com  Resource Group Admin1 (No Delete)  /subscriptions/337f2b3a-68b6-4a2e-befd-01a13f20c1d0/resourceGroups/tpl-azure-aks--mazeazusrqn9-99uz-sbx


In [52]:
!az vm list-skus \
    --location southindia \
    --resource-type virtualMachines \
    --size Standard_B \
    --all \
    --query "[?name=='Standard_B2s'].{Name:name,Restrictions:restrictions}" \
    --output table

Name
------------
Standard_B2s


In [53]:
!az aks get-versions \
    --location southindia \
    --output table

KubernetesVersion    Upgrades                                                                                                                                                                                                                   SupportPlan
-------------------  -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------  --------------------------------------
1.37.0               None available                                                                                                                                                                                                             KubernetesOfficial, AKSLongTermSupport
1.36.4               1.37.0                                                                                                                                                                   

In [54]:
RESOURCE_GROUP = "tpl-azure-aks--mazeazusrqn9-99uz-sbx"
LOCATION = "southindia"
ACR_NAME = "retailfeedbackacr2026"
AKS_NAME = "aks-retail-feedback"

In [55]:
!az acr repository show-tags \
    --name "$ACR_NAME" \
    --repository "$IMAGE_NAME" \
    --output table

Result
--------
1.0


In [56]:
!az aks list \
    --resource-group "$RESOURCE_GROUP" \
    --output table

In [57]:
!az account show \
    --query "{Subscription:name, SubscriptionId:id, State:state}" \
    --output table

Subscription          SubscriptionId                        State
--------------------  ------------------------------------  -------
Azure subscription 1  337f2b3a-68b6-4a2e-befd-01a13f20c1d0  Enabled


In [58]:
!az role assignment list \
    --resource-group "$RESOURCE_GROUP" \
    --include-inherited \
    --query "[].{Role:roleDefinitionName, Principal:principalName}" \
    --output table

usage error: --resource-group can't be an empty string. Either omit it or provide a non-empty string.


In [59]:
!az group show \
    --name "$RESOURCE_GROUP" \
    --query "{Name:name, Location:location, ProvisioningState:properties.provisioningState}" \
    --output table

In [60]:
RESOURCE_GROUP = "tpl-azure-aks--mazeazusrqn9-99uz-sbx"
LOCATION = "southindia"
ACR_NAME = "retailfeedbackacr2026"
AKS_NAME = "aks-retail-feedback"

print("Resource Group:", RESOURCE_GROUP)
print("Location:", LOCATION)
print("ACR:", ACR_NAME)
print("AKS:", AKS_NAME)

Resource Group: tpl-azure-aks--mazeazusrqn9-99uz-sbx
Location: southindia
ACR: retailfeedbackacr2026
AKS: aks-retail-feedback


In [61]:
!az group show \
    --name "$RESOURCE_GROUP" \
    --query "{Name:name, Location:location, State:properties.provisioningState}" \
    --output table

In [62]:
!az group show \
    --name "$RESOURCE_GROUP" \
    --query "{Name:name, Location:location, State:properties.provisioningState}" \
    --output table

In [63]:
print("Checking resource group:", RESOURCE_GROUP)

!az group show \
    --name "{RESOURCE_GROUP}" \
    --query "{Name:name, Location:location, State:properties.provisioningState}" \
    --output table

Checking resource group: tpl-azure-aks--mazeazusrqn9-99uz-sbx
(AuthorizationFailed) The client 'sb-mazeazusrqn9t23-ui4w@synergificsoftware.com' with object id '4f61bde3-f3f3-4ae0-b34c-21f01b67d6b3' does not have authorization to perform action 'Microsoft.Resources/subscriptions/resourcegroups/read' over scope '/subscriptions/337f2b3a-68b6-4a2e-befd-01a13f20c1d0/resourcegroups/{RESOURCE_GROUP}' or the scope is invalid. If access was recently granted, please refresh your credentials.
Code: AuthorizationFailed
Message: The client 'sb-mazeazusrqn9t23-ui4w@synergificsoftware.com' with object id '4f61bde3-f3f3-4ae0-b34c-21f01b67d6b3' does not have authorization to perform action 'Microsoft.Resources/subscriptions/resourcegroups/read' over scope '/subscriptions/337f2b3a-68b6-4a2e-befd-01a13f20c1d0/resourcegroups/{RESOURCE_GROUP}' or the scope is invalid. If access was recently granted, please refresh your credentials.


In [64]:
import os

os.environ["RG_NAME"] = RESOURCE_GROUP

!az group show \
    --name "$RG_NAME" \
    --output table

Location    Name
----------  ------------------------------------
southindia  tpl-azure-aks--mazeazusrqn9-99uz-sbx
